---
title: Глава 11. Условная логика
subtitle: SQL Lab in JupyterLab
# license: CC-BY-4.0
github: https://github.com/magus1968/learning-sql
subject: Technical Portfolio
venue: GitHub & GitVerse Pages
# abstract: |
#   В последнем запросе главы, в разделе _Этот таинственный null_, увидим причину, по которой для усечения строк в дальнейшем будет использоваться Pandas.
authors:
  - name: Alex Smirnov
    email: a@smirnovs.pro
    corresponding: true
    affiliations: Data & BI Analyst
date: 2026-10-11
abbreviations:
    MyST: Markedly Structured Text
    ПКМ: Правая Кнопка Мыши
    Jupyter Book: Инструмент сборки статических сайтов
    JupySQL: Расширение для запуска и подсветки SQL в Jupyter
    GitHub: Платформа хостинга репозиториев и совместной разработки
    GitHub Pages: Сервис бесплатного хостинга статических сайтов
    GitHub Actions: Платформа автоматизации рабочих процессов и CI/CD
    Pandas: Библиотека Python для анализа и обработки данных
    Polars: Мощный аналог Pandas на Rust/Python
---

In [ ]:
import pandas as pd
import sqlalchemy as sa
import sql

pd.set_option('display.max_rows', 20)

%load_ext sql
%config SqlMagic.displaycon = False
%config SqlMagic.autopandas = True

connection_url = sa.engine.URL.create(
    drivername='mysql+pymysql',
    host='localhost',
    port=3306,
    database='sakila',
    username='root',
    password='********',
)
engine = sa.create_engine(connection_url)

%sql engine

print(f"Pandas ver. {pd.__version__}: порог усечения строк уменьшен до 20")
print(f"SQLAlchemy ver. {sa.__version__}: подключение создано")
print(f"JupySQL ver. {sql.__version__}: подключен через SQLAlchemy Engine")

Pandas ver. 3.0.5: порог усечения строк уменьшен до 20
SQLAlchemy ver. 2.0.52: подключение создано
JupySQL ver. 0.11.1: подключен через SQLAlchemy Engine


В некоторых ситуациях требуется, чтобы SQL-логика разветвлялась в том или ином направлении в зависимости от значений определенных столбцов или выражений. Из этой главы вы узнаете, как писать операторы, которые могут вести себя по-разному в зависимости от данных, полученных во время выполнения инструкции. Механизм условной логики в инструкциях SQL – это выражение `case`, которое можно использовать в инструкциях `select`, `insert`, `update` и `delete`.

## Что такое условная логика

Условная логика – это просто возможность выбрать один из нескольких путей во время выполнения программы. Например, при запросе информации о клиенте вы можете захотеть включить столбец **customer.active**, в котором хранится **1** для обозначения активного и **0** – для обозначения неактивного клиента. Если результаты запроса используются для создания отчета, то вы можете захотеть перевести это значение в строку, чтобы улучшить удобочитаемость.

Хотя каждая база данных включает встроенные функции для подобных ситуаций, общих стандартов нет, так что нужно запоминать в какой базе данных какие функции имеются. К счастью, каждая реализация базы данных SQL включает выражение `case`, которое полезно во многих ситуациях, в том числе и для простого перевода:

:::{attention}
:class: dropdown
:open: true

В запросе автора элементы в инструкции `SELECT` расположены таким образом, что может **ошибочно** показаться как будто список полей состоит только из двух элементов **first_name, last_name**, а дальше с новой строки пошла какая-то новая крутая конструкция `CASE`, значит после **last_name** запятая не нужна.

Но колонка **last_name** НЕ последняя! \
Последней колонкой является **activity_type**, вычисленная через `CASE`.

```sql
SELECT
    first_name,          -- колонка 1 (нужна запятая)
    last_name,           -- колонка 2 (нужна запятая)
    CASE                 -- колонка 3 (НАЧАЛАСЬ)
        WHEN active = 1 THEN 'ACTIVE'
        ELSE 'INACTIVE'
    END AS activity_type -- колонка 3 (ЗАКОНЧИЛАСЬ, запятой нет)
FROM customer;
```

Вся конструкция `CASE ... END` – это **не** отдельная секция SQL вроде `FROM` или `WHERE`, это всего лишь вычисляемая колонка (такая же как `COUNT(*)` или `amount*2`).
:::

In [3]:
%%sql
SELECT first_name, last_name,
    CASE
        WHEN active = 1 THEN 'ACTIVE'
        ELSE 'INACTIVE'
    END activity_type
FROM customer;

599 rows affected.

,first_name,last_name,activity_type
0,MARY,SMITH,ACTIVE
1,PATRICIA,JOHNSON,ACTIVE
2,LINDA,WILLIAMS,ACTIVE
3,BARBARA,JONES,ACTIVE
4,ELIZABETH,BROWN,ACTIVE
...,...,...,...
594,TERRENCE,GUNDERSON,ACTIVE
595,ENRIQUE,FORSYTHE,ACTIVE
596,FREDDIE,DUGGAN,ACTIVE
597,WADE,DELVALLE,ACTIVE


Этот запрос включает выражение `case` для генерации значения столбца **activity_type**, которое возвращает строку ACTIVE или INACTIVE в зависимости от значения столбца **customer.active**.

## Выражение *case*

Все основные серверы баз данных включают встроенные функции, имитирующие инструкцию **if-then-else**, имеющуюся в большинстве языков программирования (например, в Oracle – функция `decode()`, в MySQL – `if()`, в SQL Server – `coalesce()`).

Выражения `case` также предназначены для облегчения применения логики **if-then-else**, но при этом имеют следующие преимущества перед встроенными функциями.
- Выражение `case` является частью стандарта SQL (выпуск SQL92) и реализовано в Oracle Database, SQL Server, MySQL, PostgreSQL, IBM UDB и других СУБД.
- Выражения `case` встроены в грамматику SQL и могут быть включены в инструменты `select`, `insert`, `update` и `delete`.

В следующих двух подразделах представлены два разных типа выражений `case`. Материал сопровождается некоторыми примерами case-выражений в действии.

### Поисковые выражения case

Выражение `case`, показанное ранее в этой главе, является примером поискового выражения `case`, имеющего следующий синтаксис:

```sql
CASE
    WHEN C1 THEN E1
    WHEN C2 THEN E2
    ...
    WHEN CN THEN EN
    [ELSE ED]
END
```

В этом определении символы C1, C2,... CN представляют условия (Condition), а символы E1, E2,... EN – выражения (Expression), возвращаемые выражением `case`. Если условие в предложении `when` вычисляется как истинное, выражение `case` возвращает соответствующее выражение E\#.

Кроме того, символ ED представляет выражение по умолчанию (Expression Default), которое выражение `case` возвращает, если *ни одно* из условий C1, C2,... CN не дает при вычислении `true`. Предложение `else` является необязательным, поэтому оно заключено в квадратные скобки.

Все выражения, возвращаемые различными предложениями `when` должны иметь один и тот же тип (например, `date`, `number`, `varchar`).

Вот пример поискового выражения `case`:

```sql
CASE
    WHEN 
        THEN 
    WHEN 
        THEN 
    WHEN 
        THEN 
    ELSE 
END
```

